# 01 — Corrective RAG (CRAG): Bounded Recovery

**Track:** Advanced · **Stage:** Production Patterns

Standard RAG assumes the retriever will always return the right documents. If it doesn't, the LLM hallucinates.

**Corrective RAG (CRAG)** treats retrieval as a *control system*. It adds an evaluation step *after* retrieval. If the retrieved documents are deemed irrelevant or weak, it triggers a bounded recovery action (like a web search, or query reformulation) before attempting generation.

In this comprehensive deep dive, we will:
1. **Part 1: The Theory.** Build a transparent CRAG loop from scratch in pure Python to understand the mechanics of bounded recovery.
2. **Part 2: The Production Implementation.** Migrate our logic into **LangGraph**, the industry standard for stateful AI control flows.

## Setup

We use `langchain` and `langgraph` for the production implementation.

In [ ]:
# !pip install langgraph langchain langchain-community

from typing import List, Dict, TypedDict
from langgraph.graph import StateGraph, END
from langchain_core.documents import Document
from langchain_community.llms.fake import FakeListLLM
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

---
## Part 1: The Theory of Bounded Recovery

A corrective system is not allowed to "try harder" indefinitely. Every route must have a budget (e.g., max 3 retries) and a terminal reason (e.g., "Abstained: Could not find evidence").

Let's trace a manual control loop:
`Retrieve -> Grade -> Recover (if needed) -> Verify -> Answer/Abstain`

In [ ]:
class ManualCRAG:
    def __init__(self):
        self.corpus = {
            "policy": "All employees must use the VPN when offsite."
        }
    
    def retrieve(self, query: str) -> str:
        # Simulating a bad retrieval for questions about 'cafeteria'
        if "cafeteria" in query.lower():
            return ""
        return self.corpus["policy"]
    
    def grade_evidence(self, query: str, context: str) -> bool:
        # If the context is empty, it fails the grade.
        return len(context) > 0
    
    def web_search_recovery(self, query: str) -> str:
        print("    [!] Triggering Web Search Recovery...")
        return "Web Result: The cafeteria serves tacos on Tuesday."
    
    def execute(self, query: str):
        print(f"\nQuestion: {query}")
        
        # Step 1: Retrieve
        context = self.retrieve(query)
        
        # Step 2: Grade
        is_relevant = self.grade_evidence(query, context)
        
        # Step 3: Recover
        if not is_relevant:
            print("    [X] Internal retrieval failed.")
            context = self.web_search_recovery(query)
            
        # Step 4: Answer
        print(f"    [V] Final Context used: {context}")

crag = ManualCRAG()
crag.execute("What is the offsite policy?")
crag.execute("What is for lunch in the cafeteria?")

---
## Part 2: Production Implementation with LangGraph

While the manual python class above works for simple cases, production RAG systems require:
- Asynchronous execution
- State persistence across distributed nodes
- Complex cyclical routing (e.g., retrying multiple times)

**LangGraph** solves this by defining the system as a `StateGraph`.

### Step A: Define the Graph State

The state is a `TypedDict` that is passed from node to node. Nodes return updates to this state.

In [ ]:
class GraphState(TypedDict):
    question: str
    generation: str
    documents: List[str]
    retries: int

### Step B: Define the Nodes (Actions)

Each node is a function. For this offline tutorial, we simulate the LLM and Retrieval actions.

In [ ]:
def retrieve_node(state: GraphState):
    print("---NODE: RETRIEVE---")
    question = state["question"]
    if "cafeteria" in question.lower():
        docs = ["Acme Runbook: Server maintenance at 2am."] # Completely irrelevant
    else:
        docs = ["NovaTech policy: Contractors must sign NDA."]
    return {"documents": docs}

def grade_documents_node(state: GraphState):
    print("---NODE: GRADE DOCUMENTS---")
    question = state["question"]
    docs = state["documents"]
    
    # Simulating an LLM-as-a-judge:
    if "cafeteria" in question.lower() and "maintenance" in docs[0].lower():
        print("   -> Grade: IRRELEVANT (Triggering CRAG)")
        return {"documents": []} # Clear documents because they are useless
    else:
        print("   -> Grade: RELEVANT")
        return {"documents": docs}

def web_search_node(state: GraphState):
    print("---NODE: WEB SEARCH FALLBACK---")
    return {"documents": ["Web Result: The cafeteria menu today is Taco Tuesday."]}

def generate_node(state: GraphState):
    print("---NODE: GENERATE---")
    docs = state["documents"]
    return {"generation": f"Generated Answer based on: {docs[0]}"}

### Step C: Define Edges (Routing)

Conditional edges decide which node executes next.

In [ ]:
def decide_to_generate(state: GraphState):
    print("---EDGE: ROUTING DECISION---")
    if not state["documents"]:
        print("   -> Routing to Web Search Fallback")
        return "web_search"
    else:
        print("   -> Routing to Generation")
        return "generate"

### Step D: Compile and Run

We tie the nodes and edges together into a compiled application.

In [ ]:
workflow = StateGraph(GraphState)

# Add nodes
workflow.add_node("retrieve", retrieve_node)
workflow.add_node("grade_documents", grade_documents_node)
workflow.add_node("web_search", web_search_node)
workflow.add_node("generate", generate_node)

# Add edges
workflow.set_entry_point("retrieve")
workflow.add_edge("retrieve", "grade_documents")
workflow.add_conditional_edges(
    "grade_documents",
    decide_to_generate,
    {
        "web_search": "web_search",
        "generate": "generate",
    }
)
workflow.add_edge("web_search", "generate")
workflow.add_edge("generate", END)

app = workflow.compile()

print("\n========== TEST 1: RELEVANT RETRIEVAL ==========")
inputs = {"question": "What is the contractor policy?", "retries": 0}
for output in app.stream(inputs):
    pass
print(f"\nFINAL OUTPUT: {output['generate']['generation']}\n")

print("========== TEST 2: IRRELEVANT RETRIEVAL (TRIGGERS CRAG) ==========")
inputs = {"question": "What is the cafeteria menu?", "retries": 0}
for output in app.stream(inputs):
    pass
print(f"\nFINAL OUTPUT: {output['generate']['generation']}")

## Reflection

1. **Agentic vs CRAG:** CRAG is a *deterministic* control flow. The LLM is used as a judge, but the *path* (Retrieve -> Grade -> Web Search) is hardcoded. Full Agentic RAG gives the LLM access to tools and lets it decide the path dynamically (e.g. `ReAct`). Both are valid, but CRAG is much easier to control in strict enterprise environments.
2. **Statefulness:** LangGraph provides memory. By tracking `state["retries"]`, you can prevent infinite loops if the Web Search also returns irrelevant data, enforcing a hard limit before graceful abstention.